In [4]:
# Task1_WebScraping_Sentiment.ipynb

import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
from nltk.sentiment.vader import SentimentIntensityAnalyzer



In [8]:

# Setup Selenium
driver = webdriver.Chrome()
url = "https://www.bestbuy.ca/en-ca/"
driver.get(url)
time.sleep(3)


In [9]:
reviews = []
while True:
    soup = BeautifulSoup(driver.page_source, "html.parser")
    review_blocks = soup.find_all("div", {"class":"reviewItem"})
    
    for block in review_blocks:
        title = block.find("h4").text if block.find("h4") else ""
        text = block.find("p").text if block.find("p") else ""
        rating = block.find("span", {"class":"rating"}).text if block.find("span", {"class":"rating"}) else ""
        date = block.find("span", {"class":"date"}).text if block.find("span", {"class":"date"}) else ""
        reviewer = block.find("span", {"class":"reviewer"}).text if block.find("span", {"class":"reviewer"}) else ""
        
        reviews.append([title, text, rating, date, reviewer])
    
    try:
        show_more = driver.find_element(By.XPATH, "//button[contains(text(),'Show More')]")
        show_more.click()
        time.sleep(2)
    except:
        break

driver.quit()


In [10]:
df = pd.DataFrame(reviews, columns=["Title","Review","Rating","Date","Reviewer"])


In [14]:
# Sentiment Analysis
sid = SentimentIntensityAnalyzer
df["Sentiment"] = df["Review"].apply(lambda x: sid.polarity_scores(x)["compound"])
df["Sentiment_Label"] = df["Sentiment"].apply(lambda x: "Positive" if x>0.05 else ("Negative" if x<-0.05 else "Neutral"))


In [17]:
df.to_csv("E:\\Munindhar\\HanuAi_Assignment\\data\\scraped_reviews.csv", index=False)